# arg-position-back-functions — ex3: BACK_FUNCS registry and dispatch by (fn, argnum)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `arg-position-back-functions`. Running the final beacon cell reports progress against the `Backprop: Arg-position back funcs` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Arg-position back funcs` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`arg-position-back-functions`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "arg-position-back-functions"
DD_SUBTOPIC = "Backprop: Arg-position back funcs"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `BACK_FUNCS` — dispatch back fns by (forward_fn, argnum)

Ex1 wrote `div_back0`/`div_back1`; ex2 wrote `pow_back0`/`pow_back1`. The deepening move is the GLUE: a registry `BACK_FUNCS` keyed by `(fwd_fn, argnum)` so the reverse pass can ask 'which back fn for the 1st arg of `t.divide`?' without if/elif chains.

```python
class BackFuncs:
    def __init__(self):
        self._registry = {}
    def add_back_func(self, fwd_fn, argnum, back_fn):
        self._registry[(fwd_fn, argnum)] = back_fn
    def get_back_func(self, fwd_fn, argnum):
        return self._registry[(fwd_fn, argnum)]
```

**Why `(fwd_fn, argnum)` as the key.** Each forward fn has ONE back fn per Tensor input position. `div` registers 2 entries (argnum 0 and 1); `log` registers 1 (argnum 0 only). The reverse pass iterates `recipe.parents.items()` (which is `{argnum: parent_tensor}`) and looks up `BACK_FUNCS.get_back_func(recipe.func, argnum)` for each.

**Missing-key error.** If a back fn was never registered, `KeyError` fires at reverse-pass time. ARENA's MiniTensor uses a custom message 'no back fn registered for `<fn>` at argnum `<i>`' so the failure points at the missing registration, not the dict internals.

### Exercise 3 — BACK_FUNCS registry and dispatch by (fn, argnum)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the registry-and-dispatch pattern to wire per-(fwd_fn, argnum) back fns into a `BackFuncs` table, then look them up by the same key the reverse pass uses.
> Keywords: registry, dispatch, back-funcs, argnum
> ```

**KCs targeted:** `back-funcs-registry-key-is-fn-argnum`, `missing-back-fn-raises-keyerror`

Implement `ex3_back_funcs()` returning a class `BackFuncs` and two back fns wired into an instance of it.

1. Define class `BackFuncs` with:
   - `__init__(self)` initializing `self._registry = {}` (a dict keyed by `(fwd_fn, argnum)` tuples).
   - `add_back_func(self, fwd_fn, argnum, back_fn)` storing `back_fn` at key `(fwd_fn, argnum)`.
   - `get_back_func(self, fwd_fn, argnum)` returning the stored back fn — raise the natural `KeyError` if missing (don't catch).

2. Define `div_back0(grad_out, out, x, y)` returning `grad_out / y` and `div_back1(grad_out, out, x, y)` returning `-grad_out * out / y`.

3. Instantiate `bf = BackFuncs()`, register both back fns at the correct (`t.divide`, `0` | `1`) keys, and return a dict `{'BackFuncs': BackFuncs, 'bf': bf, 'div_back0': div_back0, 'div_back1': div_back1}`.

The dispatcher uses `bf.get_back_func(t.divide, 0)` and `bf.get_back_func(t.divide, 1)` — both must succeed; lookups for unregistered keys (`t.divide, 2` or `t.add, 0`) must raise `KeyError`.

In [ ]:
def ex3_back_funcs():
    class BackFuncs:
        def __init__(self):
            self._registry = {}
        def add_back_func(self, fwd_fn, argnum, back_fn):
            self._registry[(fwd_fn, argnum)] = back_fn
        def get_back_func(self, fwd_fn, argnum):
            return self._registry[(fwd_fn, argnum)]

    def div_back0(grad_out, out, x, y):
        return grad_out / y

    def div_back1(grad_out, out, x, y):
        return -grad_out * out / y

    bf = BackFuncs()
    bf.add_back_func(t.divide, 0, div_back0)
    bf.add_back_func(t.divide, 1, div_back1)
    return {'BackFuncs': BackFuncs, 'bf': bf,
            'div_back0': div_back0, 'div_back1': div_back1}


<details><summary>Solution</summary>

```python
def ex3_back_funcs():
    class BackFuncs:
        def __init__(self):
            self._registry = {}
        def add_back_func(self, fwd_fn, argnum, back_fn):
            self._registry[(fwd_fn, argnum)] = back_fn
        def get_back_func(self, fwd_fn, argnum):
            return self._registry[(fwd_fn, argnum)]

    def div_back0(grad_out, out, x, y):
        return grad_out / y

    def div_back1(grad_out, out, x, y):
        return -grad_out * out / y

    bf = BackFuncs()
    bf.add_back_func(t.divide, 0, div_back0)
    bf.add_back_func(t.divide, 1, div_back1)
    return {'BackFuncs': BackFuncs, 'bf': bf,
            'div_back0': div_back0, 'div_back1': div_back1}
```

**Key shape `(fwd_fn, argnum)` is the whole trick.** The forward fn is the dispatching authority; argnum picks which arg's gradient. Using anything else (e.g. just the function, or a string name) breaks composition the moment you register two back fns for the same fn.

**Don't catch the KeyError inside `get_back_func`.** Let it propagate — the reverse pass needs to know if it asks for a back fn that was never registered. A friendly wrapper that re-raises with a better message is fine, but swallowing is actively harmful.

**Why a class and not just a dict.** The class wrapper is the extension point — once you add `register_kwarg_back_func`, `list_registered`, or per-instance defaults, the dict shape stops being expressive enough.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()